# 05 Receptor Consistency Check

**Scope.** Evaluate whether QM descriptor distances are aligned with public receptor-response divergence in matched structural-neighbor pairs while respecting the interpretation boundary of this orthogonal consistency analysis.

## Expected input files

- `data/receptor_annotations/receptor_response_vectors.csv`
- `data/receptor_annotations/receptor_pairwise_concordance.csv`
- `data/receptor_annotations/receptor_concordant_divergent_pairs.csv`
- `results/receptor_consistency/qm_rdkit_distance_by_receptor_group.csv`
- `results/receptor_consistency/receptor_consistency_statistics.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

SAVE_OUTPUTS = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_configs = (candidate / "configs").is_dir()
        has_readme = (candidate / "README.md").exists()
        if has_configs and has_readme:
            return candidate
        if has_configs:
            fallback = candidate
            return fallback
    raise FileNotFoundError("Could not locate project root containing configs/.")


def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def check_files(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({"path": path.as_posix(), "exists": path.exists()})
    return pd.DataFrame(rows)


def require_files(paths):
    missing = [Path(path) for path in paths if not Path(path).exists()]
    if missing:
        formatted = "
".join(f"- {path.as_posix()}" for path in missing)
        raise FileNotFoundError(
            "The following required files are missing:
"
            f"{formatted}
"
            "Data required from user later before this notebook can run end-to-end."
        )


def print_missing_file_report(report_df: pd.DataFrame) -> None:
    missing = report_df.loc[~report_df["exists"], "path"].tolist()
    if missing:
        print("Data required from user later:")
        for item in missing:
            print(f"- {item}")
    else:
        print("All expected input files are present.")


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "configs"
DATASET_CFG = load_yaml(CONFIG_DIR / "dataset.yaml")


## 1. Scope

This notebook checks processed receptor-matching outputs and grouped descriptor-distance summaries for matched structural-neighbor pairs.

## 2. Interpretation boundary

This notebook evaluates whether QM descriptor distances are aligned with public receptor-response divergence in matched structural-neighbor pairs. It does not test receptor binding mechanisms directly.

## 3. Expected inputs

The notebook expects curated public receptor-response summaries, pair-level concordance labels, and released receptor-consistency result tables.

## 4. Load configs

In [ ]:
expected_files = [
    PROJECT_ROOT / DATASET_CFG['files']['receptor_response_vectors'],
    PROJECT_ROOT / DATASET_CFG['files']['receptor_pairwise_concordance'],
    PROJECT_ROOT / DATASET_CFG['files']['receptor_concordant_divergent_pairs'],
    PROJECT_ROOT / 'results/receptor_consistency/qm_rdkit_distance_by_receptor_group.csv',
    PROJECT_ROOT / 'results/receptor_consistency/receptor_consistency_statistics.csv',
]
print('Configured receptor annotation directory:', (PROJECT_ROOT / DATASET_CFG['paths']['receptor_annotations_dir']).as_posix())

## 5. File existence checks

In [ ]:
file_report = check_files(expected_files)
display(file_report)
print_missing_file_report(file_report)

## 6. Receptor-response vector checks

In [ ]:
loaded = {}
for path in expected_files:
    if path.exists():
        loaded[path.name] = pd.read_csv(path)

if 'receptor_response_vectors.csv' in loaded:
    response_vectors = loaded['receptor_response_vectors.csv']
    display(response_vectors.head())
    print('Unique receptors:', response_vectors['receptor_id'].nunique() if 'receptor_id' in response_vectors.columns else 'column missing')
else:
    print('Receptor-response vector checks require receptor_response_vectors.csv.')

## 7. Jointly tested receptor count checks

In [ ]:
if 'receptor_pairwise_concordance.csv' in loaded:
    concordance = loaded['receptor_pairwise_concordance.csv']
    display(concordance.head())
    if 'n_jointly_tested_receptors' in concordance.columns:
        display(concordance['n_jointly_tested_receptors'].describe().to_frame(name='n_jointly_tested_receptors'))
        print('Pairs with at least two jointly tested receptors:', int((concordance['n_jointly_tested_receptors'] >= 2).sum()))
else:
    print('Jointly tested receptor count checks require receptor_pairwise_concordance.csv.')

## 8. Concordant/divergent/intermediate classification check

In [ ]:
if 'receptor_concordant_divergent_pairs.csv' in loaded:
    grouped = loaded['receptor_concordant_divergent_pairs.csv']
    display(grouped.head())
    if 'classification' in grouped.columns:
        display(grouped['classification'].value_counts(dropna=False).to_frame(name='count'))
else:
    print('Classification checks require receptor_concordant_divergent_pairs.csv.')

## 9. QM versus RDKit distance comparison

In [ ]:
distance_path = PROJECT_ROOT / 'results/receptor_consistency/qm_rdkit_distance_by_receptor_group.csv'
if distance_path.exists():
    distance_df = pd.read_csv(distance_path)
    display(distance_df.head())
    numeric_cols = [c for c in ['qm_distance', 'rdkit_distance'] if c in distance_df.columns]
    if numeric_cols:
        display(distance_df[numeric_cols].describe())
else:
    print('QM versus RDKit distance comparison requires the released distance summary table.')

## 10. Missing data checklist

In [ ]:
missing_items = file_report.loc[~file_report['exists'], 'path'].tolist()
if missing_items:
    print('Data required from user later:')
    for item in missing_items:
        print(f'- {item}')
else:
    print('No missing receptor consistency input files detected for this notebook.')